<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Deep-Learning/11-representation-self-supervised-learning.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Deep Learning guideline](Deep-Learning.html)

## **Representation Learning and Self-Supervised Learning** {#representation-self-supervised-learning}

The preceding chapters asked which architecture matches spatial, sequential, attentional, or relational structure. This chapter changes the objective: **what information should the architecture preserve in an embedding when labels are scarce, expensive, narrow, or unavailable?** Representation learning replaces manually specified features with a learned map $f_\theta:x\mapsto h$. Self-supervised learning (SSL) creates training targets from relationships already present in the data, such as two augmentations of one image, a masked region and its context, adjacent modalities, or a slowly moving teacher.

The distinction is important. Representation learning is the broad goal; self-supervision is one source of learning signal. Triplet learning may use human class labels, CLIP uses paired image-text supervision, masked autoencoding uses no class labels, and knowledge distillation transfers a teacher's outputs. All can produce embeddings, but their invariances and failure modes come from different supervision.

This chapter uses the `load_digits` copy of the [UCI Optical Recognition of Handwritten Digits dataset](https://archive.ics.uci.edu/dataset/80/optical%2Brecognition%2Bof%2B). The official [scikit-learn documentation](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html) describes 1,797 8×8 images with integer pixels from 0 to 16. UCI attributes the dataset to E. Alpaydin and C. Kaynak, assigns DOI [10.24432/C50P49](https://doi.org/10.24432/C50P49), and licenses it under CC BY 4.0. A fixed stratified 70/15/15 split is reused throughout. SSL pretraining sees only training images; class labels enter only in explicitly supervised metric learning, prompt alignment, probes, and evaluation.

![One scaled 8 by 8 handwritten digit from each class in the shared UCI-derived dataset.](assets/dl11-digits-samples.svg){fig-align="center" width="76%" fig-alt="Ten small grayscale optical digit images, one for each class from zero to nine."}

*Original data visualization generated from scikit-learn's [UCI-derived digits copy](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html), UCI dataset DOI 10.24432/C50P49, CC BY 4.0.*

### **What Makes a Useful Representation?** {#what-makes-useful-representation}

A useful representation retains factors needed by many downstream decisions while suppressing nuisance variation. For digit identity, modest sensor noise and missing pixels may be nuisances; stroke topology is usually informative. This creates two competing requirements: **invariance**, where allowed transformations leave $h$ nearly unchanged, and **selectivity**, where semantically different inputs remain distinguishable. A constant vector is perfectly invariant but useless. Raw pixels are highly selective but fragile to harmless perturbations.

Utility is conditional on a task family and deployment distribution. An embedding optimized for digit class may intentionally discard writing style, which would be harmful for writer identification. No representation is universally sufficient. Good practice therefore states the intended invariances, evaluates several downstream tasks or probes, checks robustness under realistic shifts, and measures whether information has collapsed into too few dimensions.

![PCA shows that even a simple representation defines neighborhoods and class overlap.](assets/dl11-digits-pca.svg){fig-align="center" width="76%" fig-alt="Two-dimensional PCA scatter of scaled handwritten digit pixels, colored by digit class for interpretation."}

*Original visualization of 800 scaled digit images. PCA was fit without labels; colors are added only to inspect the resulting geometry.*

<details>
<summary><strong>PyTorch: establish the shared digits split and augmentation contract</strong></summary>

```python
import copy
import math
import random

import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(1)


def seed_everything(seed=1110):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything()
digits = load_digits()
all_images = torch.tensor(digits.images, dtype=torch.float32).unsqueeze(1) / 16.0
all_labels = torch.tensor(digits.target, dtype=torch.long)
all_indices = np.arange(len(all_images))

train_idx, holdout_idx = train_test_split(
    all_indices, test_size=0.30, random_state=1110, stratify=digits.target
)
val_idx, test_idx = train_test_split(
    holdout_idx, test_size=0.50, random_state=1110,
    stratify=digits.target[holdout_idx],
)
train_images, train_labels = all_images[train_idx], all_labels[train_idx]
val_images, val_labels = all_images[val_idx], all_labels[val_idx]
test_images, test_labels = all_images[test_idx], all_labels[test_idx]


def augment_batch(images, noise_std=0.08, drop_probability=0.08):
    # Independent contrast/noise/dropout views preserve the coarse 8x8 digit shape.
    contrast = torch.empty(len(images), 1, 1, 1).uniform_(0.85, 1.15)
    noisy = images * contrast + noise_std * torch.randn_like(images)
    keep = torch.rand_like(images) > drop_probability
    return (noisy * keep).clamp(0.0, 1.0)


class DigitEncoder(nn.Module):
    def __init__(self, embedding_dim=32):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, embedding_dim),
            nn.LayerNorm(embedding_dim),
        )

    def forward(self, images):
        return self.network(images)


def make_loader(images, labels=None, batch_size=256, shuffle=True, seed=1110):
    tensors = (images,) if labels is None else (images, labels)
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        TensorDataset(*tensors), batch_size=batch_size, shuffle=shuffle,
        generator=generator,
    )


first_view = augment_batch(train_images[:16])
second_view = augment_batch(train_images[:16])
assert all_images.shape == (1797, 1, 8, 8)
assert len(set(train_idx) & set(test_idx)) == 0
assert first_view.shape == second_view.shape == (16, 1, 8, 8)
print({
    "split": (len(train_idx), len(val_idx), len(test_idx)),
    "pixel range": (float(all_images.min()), float(all_images.max())),
    "train class counts": torch.bincount(train_labels).tolist(),
})
```

</details>

The augmentation contract is deliberately modest because 8×8 digits contain little redundancy. Contrast jitter, Gaussian noise, and sparse pixel dropout usually preserve identity, whereas horizontal flips can turn the task into an invalid equivalence. Augmentation is therefore part of the learning objective: declaring two views “the same” teaches the encoder which information to remove.


### **Similarity, Distance, and Metric Learning** {#similarity-distance-metric-learning}

An embedding becomes operational through a comparison rule. Euclidean distance measures absolute displacement,

$$
d_2(h_i,h_j)=\lVert h_i-h_j\rVert_2,
$$

while cosine similarity measures angle,

$$
s_{\cos}(h_i,h_j)=\frac{h_i^\top h_j}{\lVert h_i\rVert_2\lVert h_j\rVert_2}.
$$

Cosine similarity ignores positive rescaling, which is useful when direction encodes semantics and norm reflects confidence or frequency. A learned Mahalanobis metric $d_M^2=(h_i-h_j)^\top M(h_i-h_j)$ can stretch important directions when $M\succeq0$. Neural metric learning usually moves this flexibility into the encoder and then uses Euclidean or cosine distance in the learned space.

Nearest-neighbor behavior depends jointly on representation, normalization, metric, and candidate distribution. High-dimensional distances can concentrate; approximate nearest-neighbor indexes trade exactness for latency; and a similarity threshold calibrated on one population may fail under domain shift. Retrieval quality must therefore be evaluated with the same gallery composition expected at deployment.

<details>
<summary><strong>Python: compare Euclidean and cosine neighbors on real digit pixels</strong></summary>

```python
def l2_normalize(vectors):
    return vectors / vectors.norm(dim=1, keepdim=True).clamp_min(1e-8)


query = test_images[0].flatten().unsqueeze(0)
candidate_pixels = train_images.flatten(1)
euclidean_distance = torch.cdist(query, candidate_pixels).squeeze(0)
cosine_similarity = l2_normalize(query) @ l2_normalize(candidate_pixels).T

euclidean_neighbor = int(euclidean_distance.argmin())
cosine_neighbor = int(cosine_similarity.argmax())
scaled_query = 0.1 * query
scaled_euclidean = torch.cdist(scaled_query, candidate_pixels).squeeze(0)
scaled_cosine = l2_normalize(scaled_query) @ l2_normalize(candidate_pixels).T

assert torch.allclose(cosine_similarity, scaled_cosine, atol=1e-6)
assert euclidean_distance.shape == (len(train_images),)
print({
    "query label": int(test_labels[0]),
    "Euclidean neighbor label": int(train_labels[euclidean_neighbor]),
    "cosine neighbor label": int(train_labels[cosine_neighbor]),
    "Euclidean neighbor changes after scaling": bool(
        euclidean_neighbor != int(scaled_euclidean.argmin())
    ),
})
```

</details>

Scaling the query leaves cosine ranking unchanged but can change Euclidean ranking because Euclidean distance retains magnitude. This is not proof that cosine is better. If embedding norm contains useful evidence, normalizing it destroys information; if norm is an uncontrolled artifact, normalization improves stability.


### **Siamese Networks and Triplet Loss** {#siamese-networks-triplet-loss}

A Siamese network applies one shared encoder to multiple inputs. Weight sharing ensures that an embedding coordinate means the same thing for an anchor, positive, and negative example. Pairwise contrastive loss can pull matched pairs together and push unmatched pairs beyond a margin. Triplet loss directly enforces a relative ordering:

$$
\mathcal L_{\text{triplet}}
=\max\left(0,
d(h_a,h_p)-d(h_a,h_n)+m\right),
$$

where $a$ is an anchor, $p$ shares the desired semantics, $n$ should differ, and margin $m>0$ specifies required separation. The loss is zero only when the negative is at least $m$ farther than the positive. [FaceNet](https://www.cv-foundation.org/openaccess/content_cvpr_2015/html/Schroff_FaceNet_A_Unified_2015_CVPR_paper.html) made triplet training prominent for verification and retrieval.

Triplet selection dominates optimization. Easy triplets already have zero loss and waste computation. The hardest negative may be mislabeled or an outlier and destabilize training. Semi-hard mining chooses negatives farther than the positive but still within the margin; batch-hard mining selects informative examples inside a batch. Mining must never search validation or test labels.

<details>
<summary><strong>PyTorch: train a label-supervised triplet embedding</strong></summary>

```python
def sample_triplets(images, labels, count, seed):
    generator = np.random.default_rng(seed)
    labels_np = labels.numpy()
    by_class = {label: np.flatnonzero(labels_np == label) for label in range(10)}
    anchors, positives, negatives = [], [], []
    for _ in range(count):
        anchor_index = int(generator.integers(len(images)))
        anchor_label = int(labels_np[anchor_index])
        positive_choices = by_class[anchor_label]
        positive_index = anchor_index
        while positive_index == anchor_index:
            positive_index = int(generator.choice(positive_choices))
        negative_label = int(generator.choice([x for x in range(10) if x != anchor_label]))
        negative_index = int(generator.choice(by_class[negative_label]))
        anchors.append(anchor_index)
        positives.append(positive_index)
        negatives.append(negative_index)
    return images[anchors], images[positives], images[negatives]


def triplet_satisfaction(encoder, triplets, margin=0.4):
    encoder.eval()
    with torch.no_grad():
        anchor, positive, negative = (l2_normalize(encoder(x)) for x in triplets)
        positive_distance = (anchor - positive).pow(2).sum(1)
        negative_distance = (anchor - negative).pow(2).sum(1)
    return float((positive_distance + margin < negative_distance).float().mean())


seed_everything(1112)
metric_encoder = DigitEncoder()
validation_triplets = sample_triplets(val_images, val_labels, 512, seed=1112)
before_satisfaction = triplet_satisfaction(metric_encoder, validation_triplets)
optimizer = torch.optim.AdamW(metric_encoder.parameters(), lr=2e-3, weight_decay=1e-4)
margin = 0.4
for epoch in range(35):
    metric_encoder.train()
    anchor, positive, negative = sample_triplets(
        train_images, train_labels, 1024, seed=1200 + epoch
    )
    anchor_embedding = l2_normalize(metric_encoder(anchor))
    positive_embedding = l2_normalize(metric_encoder(positive))
    negative_embedding = l2_normalize(metric_encoder(negative))
    loss = F.triplet_margin_loss(
        anchor_embedding, positive_embedding, negative_embedding, margin=margin
    )
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

after_satisfaction = triplet_satisfaction(metric_encoder, validation_triplets)
assert 0.0 <= after_satisfaction <= 1.0
print({"margin satisfaction before": round(before_satisfaction, 3),
       "after": round(after_satisfaction, 3),
       "final triplet loss": round(float(loss.detach()), 4)})
```

</details>

This example is **supervised metric learning**, not self-supervision: positive and negative identities come from digit labels. The validation statistic reports the fraction of fixed triplets satisfying a strict margin, which is more diagnostic than training loss alone. It still does not measure open-set calibration or retrieval under a different writer population.


### **Contrastive Learning and InfoNCE** {#contrastive-learning-infonce}

Contrastive SSL replaces class-defined positives with transformations or paired observations. Two independently augmented views of one sample are a positive pair; other examples in the comparison dictionary act as negatives. For anchor $i$, positive $j$, normalized projections $z$, temperature $\tau$, and candidate set $\mathcal A(i)$, InfoNCE is


$$
\ell_{i,j}=-\log
\frac{\exp(z_i^\top z_j/\tau)}
{\sum_{k\in\mathcal A(i)}\exp(z_i^\top z_k/\tau)}.
$$

The numerator rewards **alignment** of the two views. The denominator discourages every sample from collapsing to one point and promotes a spread-out representation; [Wang and Isola](https://arxiv.org/abs/2005.10242) analyze this alignment-uniformity perspective. Smaller $\tau$ sharpens competition and magnifies hard negatives, but can amplify noise and false negatives.

![The contrastive pipeline turns two augmentations into a positive pair and treats other batch samples as negatives.](assets/dl11-contrastive-pipeline.svg){fig-align="center" width="78%" fig-alt="A source image branches into two augmentations, a shared encoder, two embeddings, and an InfoNCE objective."}

*Original teaching diagram based on the [SimCLR formulation](https://proceedings.mlr.press/v119/chen20j.html) and the alignment-uniformity analysis of Wang and Isola.*

<details>
<summary><strong>PyTorch: implement NT-Xent and train a label-free SimCLR encoder</strong></summary>

```python
class SimCLR(nn.Module):
    def __init__(self, embedding_dim=32, projection_dim=16):
        super().__init__()
        self.encoder = DigitEncoder(embedding_dim)
        self.projector = nn.Sequential(
            nn.Linear(embedding_dim, 64), nn.ReLU(), nn.Linear(64, projection_dim)
        )

    def forward(self, images):
        representation = self.encoder(images)
        projection = l2_normalize(self.projector(representation))
        return representation, projection


def nt_xent_loss(first_projection, second_projection, temperature=0.2):
    batch_size = len(first_projection)
    projections = torch.cat([first_projection, second_projection], dim=0)
    similarities = projections @ projections.T / temperature
    similarities.fill_diagonal_(-torch.inf)
    positive_target = torch.arange(2 * batch_size)
    positive_target = (positive_target + batch_size) % (2 * batch_size)
    return F.cross_entropy(similarities, positive_target)


seed_everything(1113)
simclr = SimCLR()
optimizer = torch.optim.AdamW(simclr.parameters(), lr=2e-3, weight_decay=1e-4)
epoch_losses = []
for epoch in range(45):
    simclr.train()
    losses = []
    for (images,) in make_loader(train_images, shuffle=True, seed=1300 + epoch):
        first = augment_batch(images)
        second = augment_batch(images)
        _, first_projection = simclr(first)
        _, second_projection = simclr(second)
        loss = nt_xent_loss(first_projection, second_projection, temperature=0.2)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(float(loss.detach()))
    epoch_losses.append(float(np.mean(losses)))

simclr_encoder = simclr.encoder.eval()
assert np.isfinite(epoch_losses).all()
print({"InfoNCE epoch 1": round(epoch_losses[0], 4),
       "epoch 45": round(epoch_losses[-1], 4)})
```

</details>

Labels never enter this training loop. However, “label-free” does not mean assumption-free: augmentation code declares invariance, batch composition defines negatives, and the projection head determines where the contrastive loss is applied. The encoder representation is retained for downstream tasks; the smaller projection space is discarded.


### **SimCLR and MoCo** {#simclr-moco}

[SimCLR](https://proceedings.mlr.press/v119/chen20j.html) uses a shared encoder, a nonlinear projection head, two strong views, and in-batch negatives. With batch size $B$, the symmetric loss compares $2B$ projections. Its simplicity comes with a systems coupling: a larger negative dictionary requires a larger batch or cross-device communication.

[Momentum Contrast (MoCo)](https://arxiv.org/abs/1911.05722) decouples dictionary size from batch size. A query encoder receives gradients; a key encoder follows it by exponential moving average (EMA),

$$
\theta_k\leftarrow \mu\theta_k+(1-\mu)\theta_q,
$$

and a FIFO queue retains recent normalized keys as negatives. Large $\mu$ makes keys change slowly, keeping queued representations mutually consistent. A stale queue is still a trade-off: it increases diversity but contains embeddings produced by older encoders.

![SimCLR, MoCo, BYOL, and DINO use different mechanisms to create targets and avoid collapse.](assets/dl11-ssl-families.svg){fig-align="center" width="78%" fig-alt="Four panels comparing SimCLR in-batch negatives, MoCo momentum queue, BYOL prediction, and DINO teacher distributions."}

*Original comparison diagram based on the [SimCLR](https://proceedings.mlr.press/v119/chen20j.html), [MoCo](https://arxiv.org/abs/1911.05722), [BYOL](https://proceedings.neurips.cc/paper/2020/hash/f3ada80d5c4ee70142b17b8192b2958e-Abstract.html), and [DINO](https://arxiv.org/abs/2104.14294) papers.*

<details>
<summary><strong>PyTorch: build a momentum encoder and circular negative queue</strong></summary>

```python
seed_everything(1114)
query_encoder = copy.deepcopy(simclr_encoder)
key_encoder = copy.deepcopy(query_encoder)
for parameter in key_encoder.parameters():
    parameter.requires_grad_(False)

queue_size = 256
queue = l2_normalize(torch.randn(queue_size, 32))
queue_pointer = 0
query_optimizer = torch.optim.AdamW(query_encoder.parameters(), lr=8e-4)
momentum = 0.99

for step in range(24):
    indices = torch.randperm(len(train_images))[:64]
    query_view = augment_batch(train_images[indices])
    key_view = augment_batch(train_images[indices])
    queries = l2_normalize(query_encoder(query_view))
    with torch.no_grad():
        keys = l2_normalize(key_encoder(key_view))
    positive_logit = (queries * keys).sum(1, keepdim=True)
    negative_logits = queries @ queue.T
    logits = torch.cat([positive_logit, negative_logits], dim=1) / 0.2
    loss = F.cross_entropy(logits, torch.zeros(len(queries), dtype=torch.long))
    query_optimizer.zero_grad()
    loss.backward()
    query_optimizer.step()

    with torch.no_grad():
        for query_parameter, key_parameter in zip(
            query_encoder.parameters(), key_encoder.parameters()
        ):
            key_parameter.mul_(momentum).add_(query_parameter, alpha=1 - momentum)
        end = queue_pointer + len(keys)
        if end <= queue_size:
            queue[queue_pointer:end] = keys
        else:
            first_count = queue_size - queue_pointer
            queue[queue_pointer:] = keys[:first_count]
            queue[:end - queue_size] = keys[first_count:]
        queue_pointer = end % queue_size

assert queue.shape == (256, 32) and not queue.requires_grad
assert 0 <= queue_pointer < queue_size
print({"dictionary entries": queue_size, "batch queries": len(queries),
       "queue pointer": queue_pointer, "last MoCo loss": round(float(loss), 4)})
```

</details>

The queue is detached from gradients, and keys are created by the slowly updated encoder. Enqueuing query embeddings with gradients or updating the key encoder by ordinary backpropagation changes the algorithm. In distributed training, queues and batch statistics must also remain consistent across workers.


### **Non-Contrastive Learning: BYOL and DINO** {#non-contrastive-learning-byol-dino}

Negative-free methods ask whether two views can agree without explicitly repelling other samples. [BYOL](https://proceedings.neurips.cc/paper/2020/hash/f3ada80d5c4ee70142b17b8192b2958e-Abstract.html) has an online encoder-projector-predictor and an EMA target encoder-projector. The online branch predicts the stop-gradient target representation of the other view, and the directions are swapped. Architectural asymmetry, stop-gradient, normalization, optimization, augmentation, and the moving target interact to avoid trivial collapse; “EMA alone prevents collapse” is too strong a claim.

[DINO](https://arxiv.org/abs/2104.14294) matches a student's probability distribution to an EMA teacher's distribution across views. Teacher logits are centered to prevent one dimension from dominating and sharpened with a lower temperature to create informative targets. Multi-crop augmentation gives the student local and global views while the teacher sees global views. DINO's emergent attention behavior is associated with its full Vision Transformer setup, not guaranteed by any teacher-student loss.

<details>
<summary><strong>PyTorch: train BYOL regression and construct a DINO-style target</strong></summary>

```python
class BYOLProjector(nn.Module):
    def __init__(self, input_dim=32, output_dim=32):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Linear(64, output_dim),
        )

    def forward(self, features):
        return self.network(features)


def cosine_regression(prediction, target):
    prediction = l2_normalize(prediction)
    target = l2_normalize(target.detach())
    return 2 - 2 * (prediction * target).sum(1).mean()


seed_everything(1115)
online_encoder = DigitEncoder()
online_projector = BYOLProjector()
predictor = nn.Sequential(nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 32))
target_encoder = copy.deepcopy(online_encoder)
target_projector = copy.deepcopy(online_projector)
for module in (target_encoder, target_projector):
    for parameter in module.parameters():
        parameter.requires_grad_(False)

optimizer = torch.optim.AdamW(
    list(online_encoder.parameters()) + list(online_projector.parameters())
    + list(predictor.parameters()), lr=1.5e-3, weight_decay=1e-4
)
byol_losses = []
for epoch in range(30):
    losses = []
    for (images,) in make_loader(train_images, shuffle=True, seed=1400 + epoch):
        first, second = augment_batch(images), augment_batch(images)
        first_prediction = predictor(online_projector(online_encoder(first)))
        second_prediction = predictor(online_projector(online_encoder(second)))
        with torch.no_grad():
            first_target = target_projector(target_encoder(first))
            second_target = target_projector(target_encoder(second))
        loss = 0.5 * (
            cosine_regression(first_prediction, second_target)
            + cosine_regression(second_prediction, first_target)
        )
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        with torch.no_grad():
            for online, target in zip(online_encoder.parameters(), target_encoder.parameters()):
                target.mul_(0.99).add_(online, alpha=0.01)
            for online, target in zip(online_projector.parameters(), target_projector.parameters()):
                target.mul_(0.99).add_(online, alpha=0.01)
        losses.append(float(loss.detach()))
    byol_losses.append(float(np.mean(losses)))

byol_encoder = online_encoder.eval()
# DINO uses centered, sharpened teacher distributions instead of vector regression.
student_head = nn.Linear(32, 10)
teacher_head = copy.deepcopy(student_head)
with torch.no_grad():
    teacher_logits = teacher_head(target_encoder(second_view))
    center = teacher_logits.mean(0, keepdim=True)
    teacher_probabilities = torch.softmax((teacher_logits - center) / 0.04, dim=1)
student_log_probabilities = torch.log_softmax(
    student_head(online_encoder(first_view)) / 0.1, dim=1
)
dino_style_loss = -(teacher_probabilities * student_log_probabilities).sum(1).mean()

assert torch.allclose(teacher_probabilities.sum(1), torch.ones(16), atol=1e-6)
print({"BYOL epoch 1 -> 30": (round(byol_losses[0], 4), round(byol_losses[-1], 4)),
       "DINO-style distribution loss": round(float(dino_style_loss), 4)})
```

</details>

The tiny digits experiment implements the mechanisms rather than reproducing ImageNet results. A decreasing BYOL loss only shows that online predictions match target features; collapse diagnostics and downstream probes are still required. The DINO code demonstrates centering, sharpening, and cross-entropy against a stop-gradient distribution, but does not claim to be a full multi-crop ViT training run.


### **Cross-Modal Alignment and CLIP** {#cross-modal-alignment-clip}

Cross-modal learning defines positives from paired modalities. [CLIP](https://proceedings.mlr.press/v139/radford21a.html) uses separate image and text encoders, normalizes both outputs, and forms a batch similarity matrix

$$
S_{ij}=\exp(s)\,\frac{v_i^\top t_j}{\lVert v_i\rVert\lVert t_j\rVert},
$$

where learned logit scale $s$ acts as inverse temperature. Symmetric cross-entropy asks each image to retrieve its text and each text to retrieve its image. At inference, natural-language prompts become class prototypes or retrieval queries, connecting representation learning to zero-shot transfer.

![Dual encoders align images and text in one normalized similarity space.](assets/dl11-cross-modal.svg){fig-align="center" width="75%" fig-alt="Image and text towers produce embeddings that are compared in a similarity matrix with matched pairs on the diagonal."}

*Original teaching diagram based on [Learning Transferable Visual Models From Natural Language Supervision](https://proceedings.mlr.press/v139/radford21a.html).*

The digits dataset has class labels, not natural captions. The code therefore maps each class to an English prompt such as `digit seven` and learns ten text prototypes. This is **label-supervised cross-modal alignment**, not a faithful CLIP pretraining corpus. It isolates the dual-encoder geometry while making the supervision boundary explicit.

<details>
<summary><strong>PyTorch: align digit images with ten text prompts</strong></summary>

```python
digit_prompts = [
    "digit zero", "digit one", "digit two", "digit three", "digit four",
    "digit five", "digit six", "digit seven", "digit eight", "digit nine",
]


class TinyDualEncoder(nn.Module):
    def __init__(self, embedding_dim=32):
        super().__init__()
        self.image_encoder = DigitEncoder(embedding_dim)
        self.text_embeddings = nn.Embedding(len(digit_prompts), embedding_dim)
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07)))

    def forward(self, images):
        image_features = l2_normalize(self.image_encoder(images))
        prompt_ids = torch.arange(len(digit_prompts))
        text_features = l2_normalize(self.text_embeddings(prompt_ids))
        scale = self.logit_scale.exp().clamp(max=100)
        return scale * image_features @ text_features.T


seed_everything(1116)
dual_encoder = TinyDualEncoder()
optimizer = torch.optim.AdamW(dual_encoder.parameters(), lr=2e-3, weight_decay=1e-4)
for epoch in range(45):
    for images, labels in make_loader(
        train_images, train_labels, shuffle=True, seed=1500 + epoch
    ):
        logits = dual_encoder(augment_batch(images, noise_std=0.04, drop_probability=0.04))
        loss = F.cross_entropy(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

dual_encoder.eval()
with torch.no_grad():
    validation_logits = dual_encoder(val_images)
    validation_accuracy = (validation_logits.argmax(1) == val_labels).float().mean()
clip_image_encoder = dual_encoder.image_encoder
assert validation_logits.shape == (len(val_images), 10)
print({"prompts": digit_prompts[:3] + ["..."],
       "validation prompt accuracy": round(float(validation_accuracy), 3),
       "learned temperature": round(float(dual_encoder.logit_scale.exp().reciprocal()), 4)})
```

</details>

Real web-scale pairs are noisy, duplicated, culturally uneven, and not independent. Batch negatives may include valid alternative captions; prompt wording changes scores; and zero-shot labels inherit biases from paired data. Cross-modal scale increases coverage, but it does not turn noisy text into an objective semantic ground truth.


### **Masked Representation Learning** {#masked-representation-learning}

Masked learning removes part of an observation and predicts it from visible context. For mask $M\in\{0,1\}^D$, a reconstruction objective can evaluate only hidden coordinates,

$$
\mathcal L_{\text{mask}}=
\frac{1}{\sum_d M_d}\sum_{d=1}^{D}M_d\left(x_d-\hat x_d\right)^2.
$$

Language models predict masked tokens or next tokens; image models predict pixels, discrete visual tokens, or latent targets. [Masked Autoencoders (MAE)](https://arxiv.org/abs/2111.06377) use an asymmetric design in which a heavy encoder processes only visible patches and a lighter decoder reconstructs missing pixels. High masking ratios reduce encoder cost and stop the task from becoming local copying when images contain substantial redundancy.

![A masked encoder must summarize visible evidence so a decoder can reconstruct hidden coordinates.](assets/dl11-masked-learning.svg){fig-align="center" width="76%" fig-alt="Original image, binary mask, encoder latent, decoder reconstruction, and loss restricted to masked positions."}

*Original teaching diagram based on the [Masked Autoencoders](https://arxiv.org/abs/2111.06377) objective.*

<details>
<summary><strong>PyTorch: train a masked digit autoencoder without labels</strong></summary>

```python
class MaskedDigitAutoencoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(64, 96), nn.ReLU(), nn.Linear(96, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 96), nn.ReLU(), nn.Linear(96, 64), nn.Sigmoid()
        )

    def forward(self, flattened, mask):
        masked_input = torch.where(mask, torch.full_like(flattened, -1.0), flattened)
        latent = self.encoder(masked_input)
        return self.decoder(latent), latent


seed_everything(1117)
masked_autoencoder = MaskedDigitAutoencoder()
optimizer = torch.optim.AdamW(masked_autoencoder.parameters(), lr=2e-3)
mask_ratio = 0.45
for epoch in range(55):
    for (images,) in make_loader(train_images, shuffle=True, seed=1600 + epoch):
        flattened = images.flatten(1)
        mask = torch.rand_like(flattened) < mask_ratio
        reconstruction, _ = masked_autoencoder(flattened, mask)
        loss = F.mse_loss(reconstruction[mask], flattened[mask])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

test_flattened = test_images.flatten(1)
generator = torch.Generator().manual_seed(1117)
test_mask = torch.rand(test_flattened.shape, generator=generator) < mask_ratio
masked_autoencoder.eval()
with torch.no_grad():
    test_reconstruction, masked_embeddings = masked_autoencoder(test_flattened, test_mask)
    reconstruction_mse = F.mse_loss(
        test_reconstruction[test_mask], test_flattened[test_mask]
    )
    train_pixel_mean = train_images.flatten(1).mean(0)
    baseline_mse = F.mse_loss(
        train_pixel_mean.expand_as(test_flattened)[test_mask], test_flattened[test_mask]
    )

assert masked_embeddings.shape == (len(test_images), 32)
print({"masked fraction": round(float(test_mask.float().mean()), 3),
       "model masked MSE": round(float(reconstruction_mse), 4),
       "train-mean baseline MSE": round(float(baseline_mse), 4)})
```

</details>

The `-1` mask sentinel lies outside the valid pixel range, so the encoder can distinguish a hidden zero from a visible background zero. The model is compared with a train-set pixel-mean reconstruction to prevent a low MSE from looking meaningful by itself. Reconstruction quality and representation utility are not identical: a decoder may reward texture detail that a classifier should ignore.


### **Knowledge Distillation** {#knowledge-distillation}

Knowledge distillation trains a smaller student to reproduce a teacher's predictive distribution or intermediate representations. For class logits $z_t,z_s$, temperature $T$, hard label $y$, and mixing coefficient $\alpha$,

$$
\mathcal L=(1-\alpha)\operatorname{CE}(y,z_s)
+\alpha T^2\operatorname{KL}\!\left(
\operatorname{softmax}(z_t/T)\;\Vert\;
\operatorname{softmax}(z_s/T)
\right).
$$

Temperature reveals relative probabilities among non-target classes, sometimes called “dark knowledge.” The $T^2$ factor compensates for gradient shrinkage introduced by softened probabilities. [Hinton, Vinyals, and Dean](https://arxiv.org/abs/1503.02531) developed this formulation for compressing ensembles and large networks.

Distillation is not limited to final logits. Feature distillation aligns hidden states, attention transfer aligns maps, and self-distillation uses another checkpoint or EMA branch of the same architecture. A student cannot be expected to exceed information absent from the teacher and data; it can also inherit teacher calibration errors and bias.

<details>
<summary><strong>PyTorch: distill the digit prompt teacher into a smaller student</strong></summary>

```python
for parameter in dual_encoder.parameters():
    parameter.requires_grad_(False)


class SmallStudent(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(), nn.Linear(64, 24), nn.ReLU(), nn.Linear(24, 10)
        )

    def forward(self, images):
        return self.network(images)


def train_student(use_distillation, seed):
    seed_everything(seed)
    student = SmallStudent()
    optimizer = torch.optim.AdamW(student.parameters(), lr=2e-3)
    temperature, soft_weight = 3.0, 0.7
    for epoch in range(45):
        for images, labels in make_loader(
            train_images, train_labels, shuffle=True, seed=1700 + epoch
        ):
            student_logits = student(images)
            hard_loss = F.cross_entropy(student_logits, labels)
            if use_distillation:
                with torch.no_grad():
                    teacher_logits = dual_encoder(images)
                soft_loss = F.kl_div(
                    F.log_softmax(student_logits / temperature, dim=1),
                    F.softmax(teacher_logits / temperature, dim=1),
                    reduction="batchmean",
                ) * temperature**2
                loss = (1 - soft_weight) * hard_loss + soft_weight * soft_loss
            else:
                loss = hard_loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    student.eval()
    with torch.no_grad():
        accuracy = (student(test_images).argmax(1) == test_labels).float().mean()
    return student, float(accuracy)


hard_student, hard_accuracy = train_student(False, 1118)
distilled_student, distilled_accuracy = train_student(True, 1118)
teacher_parameters = sum(parameter.numel() for parameter in dual_encoder.parameters())
student_parameters = sum(parameter.numel() for parameter in distilled_student.parameters())

assert student_parameters < teacher_parameters
print({"teacher/student parameters": (teacher_parameters, student_parameters),
       "hard-label accuracy": round(hard_accuracy, 3),
       "distilled accuracy": round(distilled_accuracy, 3)})
```

</details>

The hard-only and distilled students start from the same seed and use the same split. One tiny run need not show a distillation gain, so the code reports rather than asserts superiority. The valid conclusion is that soft teacher probabilities can be combined with hard labels while reducing parameter count; latency and memory should still be measured on the deployment hardware.


### **Embedding Retrieval and Representation Evaluation** {#embedding-retrieval-representation-evaluation}

SSL loss is a pretraining diagnostic, not the final measure of representation quality. Evaluation should freeze the encoder and ask whether simple downstream mechanisms can use its output. A **linear probe** trains one linear classifier, testing approximate linear accessibility of labels. A $k$-nearest-neighbor classifier tests local geometry without fitting a deep head. Retrieval reports Recall@$K$, precision, mAP, or NDCG according to whether one or many relevant candidates exist.

Protocol details matter. Probe hyperparameters must be selected on validation data; test labels are touched once. Every representation uses the same split and preprocessing. A strong nonlinear probe can hide a weak representation by relearning the task, while a tiny linear probe may underfit. Fine-tuning measures adaptation performance rather than frozen representation quality and belongs to the next chapter.

<details>
<summary><strong>Python: run a validation-selected linear probe, 5-NN, and retrieval</strong></summary>

```python
def encode_in_batches(encoder, images, batch_size=256):
    encoder.eval()
    outputs = []
    with torch.no_grad():
        for start in range(0, len(images), batch_size):
            outputs.append(encoder(images[start:start + batch_size]))
    return torch.cat(outputs)


train_embeddings = l2_normalize(encode_in_batches(simclr_encoder, train_images)).numpy()
val_embeddings = l2_normalize(encode_in_batches(simclr_encoder, val_images)).numpy()
test_embeddings = l2_normalize(encode_in_batches(simclr_encoder, test_images)).numpy()

# Select the linear-probe regularization on validation data, then touch test once.
best_validation_accuracy, best_probe = -1.0, None
for regularization in (0.1, 1.0, 10.0):
    probe = LogisticRegression(C=regularization, max_iter=1000, random_state=1119)
    probe.fit(train_embeddings, train_labels.numpy())
    score = probe.score(val_embeddings, val_labels.numpy())
    if score > best_validation_accuracy:
        best_validation_accuracy, best_probe = score, probe
linear_probe_accuracy = best_probe.score(test_embeddings, test_labels.numpy())

knn = KNeighborsClassifier(n_neighbors=5, metric="cosine", weights="distance")
knn.fit(train_embeddings, train_labels.numpy())
knn_accuracy = knn.score(test_embeddings, test_labels.numpy())

similarities = test_embeddings @ train_embeddings.T
ranking = np.argsort(-similarities, axis=1)
recall_at_1 = np.mean(
    train_labels.numpy()[ranking[:, :1]] == test_labels.numpy()[:, None]
)
recall_at_5 = np.mean(
    np.any(train_labels.numpy()[ranking[:, :5]] == test_labels.numpy()[:, None], axis=1)
)

assert 0 <= linear_probe_accuracy <= 1 and recall_at_5 >= recall_at_1
print({"linear probe validation/test": (round(best_validation_accuracy, 3), round(linear_probe_accuracy, 3)),
       "5-NN test accuracy": round(knn_accuracy, 3),
       "retrieval Recall@1/5": (round(float(recall_at_1), 3), round(float(recall_at_5), 3))})
```

</details>

Recall@1 here asks whether the nearest training image has the same digit class; Recall@5 asks whether any of the five nearest candidates does. This class-based relevance definition is suitable for a teaching dataset but differs from instance retrieval, where another image of the same class may still be considered wrong. Evaluation must define relevance before choosing a metric.


### **Failure Modes and Shortcut Learning** {#failure-modes-shortcut-learning}

Representation learning can optimize its pretext objective while learning the wrong invariance. If positive views share a border, watermark, acquisition device, or preprocessing artifact, the encoder may match that shortcut instead of semantic content. If augmentation is too weak, instance identity is trivial; if too strong, positives no longer preserve the target concept. Domain knowledge is required to define valid transformations.

Contrastive methods face **false negatives** when semantically related samples are treated as different instances. Small dictionaries provide weak competition; extremely hard negatives may be mislabeled. Non-contrastive methods face **representation collapse**, where all inputs map to one vector, and **dimensional collapse**, where variance survives in only a few directions. Feature standard deviation, covariance spectrum, effective rank, pairwise cosine similarity, and downstream probes reveal different aspects of collapse.

Other shortcuts enter through evaluation: fitting normalization on all data, selecting checkpoints on test retrieval, placing near-duplicate writers across splits, or using labels to design “self-supervised” pairs without disclosure. Provenance assertions should accompany metric code.

<details>
<summary><strong>PyTorch: diagnose collapse, alignment, and false negatives</strong></summary>

```python
def representation_diagnostics(embeddings):
    centered = embeddings - embeddings.mean(0, keepdim=True)
    covariance = centered.T @ centered / max(len(embeddings) - 1, 1)
    eigenvalues = torch.linalg.eigvalsh(covariance).clamp_min(0)
    probabilities = eigenvalues / eigenvalues.sum().clamp_min(1e-12)
    effective_rank = torch.exp(
        -(probabilities * probabilities.clamp_min(1e-12).log()).sum()
    )
    normalized = l2_normalize(embeddings)
    off_diagonal_similarity = (
        (normalized @ normalized.T).sum() - len(embeddings)
    ) / (len(embeddings) * (len(embeddings) - 1))
    return {
        "mean feature std": float(embeddings.std(0).mean()),
        "effective rank": float(effective_rank),
        "mean off-diagonal cosine": float(off_diagonal_similarity),
    }


with torch.no_grad():
    simclr_test = simclr_encoder(test_images[:128])
    byol_test = byol_encoder(test_images[:128])
    collapsed = torch.ones_like(simclr_test)
    view_one = l2_normalize(simclr_encoder(augment_batch(test_images[:128])))
    view_two = l2_normalize(simclr_encoder(augment_batch(test_images[:128])))
    positive_alignment = (view_one * view_two).sum(1).mean()

batch_labels = test_labels[:128]
same_semantic_class = batch_labels[:, None] == batch_labels[None, :]
false_negative_rate = float(
    (same_semantic_class & ~torch.eye(len(batch_labels), dtype=torch.bool)).float().mean()
)

diagnostics = {
    "SimCLR": representation_diagnostics(simclr_test),
    "BYOL": representation_diagnostics(byol_test),
    "collapsed control": representation_diagnostics(collapsed),
}
assert diagnostics["collapsed control"]["effective rank"] <= 1.01
print({name: {key: round(value, 3) for key, value in report.items()}
       for name, report in diagnostics.items()})
print({"positive-view cosine": round(float(positive_alignment), 3),
       "same-class pairs treated as negatives": round(false_negative_rate, 3)})
```

</details>

The all-ones control has effective rank near one and exposes what complete collapse looks like. Healthy effective rank alone is insufficient: random noise can occupy every dimension yet encode no useful semantics. Conversely, a task may genuinely need a low-dimensional representation. Collapse diagnostics, pretext loss, linear probes, retrieval, robustness, and transfer evidence must be interpreted together.


### **Chapter Comparison and Summary** {#chapter-comparison-summary}

Representation learning is supervision design expressed through geometry. Positive pairs determine what should become invariant; negatives or anti-collapse mechanisms determine what must remain distinct; architecture and projection heads determine where information can flow; evaluation determines whether the geometry is useful beyond the pretext task.

| Method | Training signal | Collapse/competition mechanism | Main strength | Main risk |
|---|---|---|---|---|
| Siamese/triplet | Labeled positive and negative identities | Explicit margin against selected negatives | Directly optimizes verification/retrieval ordering | Mining bias, mislabeled hard negatives, label cost |
| SimCLR/InfoNCE | Two augmented views of each instance | In-batch negatives and normalized temperature-scaled logits | Simple, strong, transparent objective | Batch-size coupling and false negatives |
| MoCo | Positive views plus queued keys | Large FIFO dictionary and momentum key encoder | Large negative set with modest batch size | Stale keys and distributed queue consistency |
| BYOL | Online prediction of EMA target view | Stop-gradient, predictor, EMA, normalization, augmentation | No explicit negatives | Collapse must be diagnosed; mechanism interactions matter |
| DINO | Student distribution matches centered/sharpened EMA teacher | Centering, sharpening, EMA, multi-crop design | Strong self-distilled semantic features | Sensitive schedules; emergent behavior is setup dependent |
| CLIP-style alignment | Paired image and text observations | Cross-modal batch competition | Retrieval and prompt-based transfer across modalities | Pair noise, prompt sensitivity, cultural and exposure bias |
| Masked modeling | Reconstruct hidden tokens, pixels, or latents | Information bottleneck created by masking | Scalable use of unlabeled context | Reconstruction detail may not equal semantic utility |
| Knowledge distillation | Teacher logits or hidden features | Teacher distribution constrains student | Compression and knowledge transfer | Student inherits teacher errors and bias |

The shared digits experiment enforced an explicit supervision ledger. Triplet sampling used labels and was called supervised. SimCLR, MoCo, BYOL, and masked reconstruction used only training images. Prompt alignment used digit names and was called label supervised. Linear probes and retrieval used labels only after the encoder was frozen. That ledger is more informative than labeling every method simply “unsupervised.”

A practical workflow is:

1. define downstream tasks and valid invariances before choosing augmentations;
2. establish raw-feature, random-encoder, and supervised baselines;
3. choose contrastive, teacher-student, masked, or cross-modal targets according to available structure;
4. monitor pretext loss together with variance, effective rank, alignment, and uniformity;
5. freeze the encoder and evaluate linear probes, $k$-NN, retrieval, calibration, and robustness on fixed splits;
6. audit labels, pairs, negatives, augmentations, duplicate entities, and test access;
7. only then decide whether full fine-tuning or parameter-efficient adaptation is justified.

The next chapter builds on this foundation: once a representation has been pretrained and evaluated, transfer learning determines how much of it to freeze, update, or adapt for a new domain and task.
